## Model Training

**1.1 Import Data and Required Packages**

Importing Pandas, Numpy, Matplotlib, Seaborn and warings Library

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.model_selection import RandomizedSearchCV, train_test_split

import warnings
warnings.filterwarnings('ignore')

**Import the CSV Data as pandas DataFrame**

In [2]:
df = pd.read_csv('data/Final_data.csv')

***Show Top 5 Records***

In [3]:
df.head()

,category,commodity,unit,usdprice,is_war,month,day
0,0,22,12,9.52,0,1,15
1,0,23,12,17.86,0,1,15
2,1,9,1,33.81,0,1,15
3,1,10,2,20.83,0,1,15
4,1,14,12,28.57,0,1,15


**Preparing X and Y variables**

In [8]:
X = df.drop(columns=['usdprice', 'day'])
y = df['usdprice']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
X_train.shape, X_test.shape

((248, 5), (63, 5))

**Creat an Evaluation Function to give all metrics after model training**

In [15]:
def evaluate_model(true, predict):
    mae = mean_absolute_error(true, predict)
    mse = mean_squared_error(true, predict)
    rmse = np.sqrt(mse)
    r2_scores = r2_score(true, predict)
    return mae, rmse, r2_scores

In [17]:
models = {
    "XGBRegressor": XGBRegressor(),
    "Random Forest Regressor": RandomForestRegressor()
}
model_list = []
r2_list = []
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])

    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print("-------------------------")

    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    r2_list.append(model_test_r2)
    
    print('='*35)
    print('\n')

XGBRegressor
Model performance for Training set
- Root Mean Squared Error: 2.9008
- Mean Absolute Error: 0.8407
- R2 Score: 0.9887
-------------------------
Model performance for Test set
- Root Mean Squared Error: 13.1053
- Mean Absolute Error: 8.5452
- R2 Score: 0.7037


Random Forest Regressor
Model performance for Training set
- Root Mean Squared Error: 8.0198
- Mean Absolute Error: 3.9939
- R2 Score: 0.9140
-------------------------
Model performance for Test set
- Root Mean Squared Error: 12.9880
- Mean Absolute Error: 8.7833
- R2 Score: 0.7089




### 📊 Model Evaluation and Performance Analysis

* **Model Comparison:**
    - **Random Forest Regressor:** Achieved a Test **R2 Score of 70.89%**, demonstrating a balanced performance between training (91.40%) and testing.
    - **XGBRegressor:** Delivered a Test **R2 Score of 70.37%**, but showed heavy reliance on training patterns with a 98.87% score.

* **Key Observations:**
    - **Overfitting Check:** The **XGBoost** model exhibits significant overfitting (98% Train vs 70% Test). It has "memorized" the training noise rather than learning the general trends.
    - **Model Reliability:** **Random Forest** proved to be more **robust** and reliable for this dataset. The smaller gap between training and testing indicates better generalization to unseen market prices.
    - **Error Analysis:** Both models yielded a **Root Mean Squared Error (RMSE)** of ~13.0, which is expected given the high price volatility and inflation spikes present in the war-time data.

* **Final Conclusion:**
    - **Random Forest Regressor** is selected as the final model for the production pipeline due to its stability and superior performance on the testing set.

**Fitting the Model**

In [18]:
final_model = RandomForestRegressor()
final_model.fit(X_train, y_train)

RandomForestRegressor()

**Save the Model**

In [20]:
import os
import pickle

output_dir = os.path.join("..", "artifacts")
model_path = os.path.join(output_dir, "model.pkl")

os.makedirs(output_dir, exist_ok=True)

with open(model_path, "wb") as file_obj:
    pickle.dump(final_model, file_obj)

print(f"Model saved successfully at: {os.path.abspath(model_path)}")

Model saved successfully at: d:\Iran_food_prices\artifacts\model.pkl
